# 03 — Evaluate ModernBERT on frozen LitSearch pools

This notebook reranks exactly the first 100 documents in each saved
BM25, GRIT, and Qwen3 candidate file. It does not retrieve new
candidates. It uses the pinned LitSearch corpus revision and writes
the repository's normal JSONL schema (`pre_reranked`, `retrieved`).

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers.cross_encoder import CrossEncoder

from tools.modernbert_supervised_baseline import (
    read_jsonl,
    rerank_litsearch_rows,
    reproducibility_manifest,
    sha256_file,
    write_json,
    write_jsonl,
)

ARTIFACT_DIR = Path(os.environ.get(
    "MODERNBERT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "modernbert_supervised_baseline"),
))
MODEL_DIR = ARTIFACT_DIR / "models" / "modernbert_base_dorismae_lambda_seed3407" / "final"
OUTPUT_DIR = ARTIFACT_DIR / "litsearch"
INPUT_FILES = {
    "bm25": REPO_ROOT / "LitSearch.title_abstract.bm25.jsonl",
    "grit": REPO_ROOT / "LitSearch.title_abstract.grit.jsonl",
    "qwen3": REPO_ROOT / "LitSearch.title_abstract.qwen3.jsonl",
}
DATASET_ID = "princeton-nlp/LitSearch"
DATASET_REVISION = "9573fb2"
RERANK_DEPTH = 100
MAX_LENGTH = 1024
PREDICT_BATCH_SIZE = 64
EXPECTED_QUERIES = 597

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Run notebook 02 first; missing {MODEL_DIR}")
for name, path in INPUT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} candidate file: {path}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
corpus = load_dataset(
    DATASET_ID,
    "corpus_clean",
    split="full",
    revision="9573fb2",
)
corpus_lookup = {
    int(item["corpusid"]): (
        f"Title: {item['title']}\nAbstract: {item['abstract']}"
    )
    for item in corpus
}
model = CrossEncoder(str(MODEL_DIR), max_length=MAX_LENGTH)

def score_pairs(pairs):
    return model.predict(
        pairs,
        batch_size=PREDICT_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

print(f"Corpus papers: {len(corpus_lookup):,}")

In [ ]:
run_records = {}
for retriever, input_path in INPUT_FILES.items():
    rows = read_jsonl(input_path)
    assert len(rows) == EXPECTED_QUERIES
    assert all(len(row["retrieved"]) >= RERANK_DEPTH for row in rows)

    # Freeze the same top-100 candidate pool used by the LLM comparisons.
    rows = [
        {**row, "retrieved": [int(value) for value in row["retrieved"][:RERANK_DEPTH]]}
        for row in rows
    ]
    input_sets = [set(row["retrieved"]) for row in rows]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    reranked = rerank_litsearch_rows(rows, corpus_lookup, score_pairs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    assert all(
        set(row["retrieved"]) == expected
        for row, expected in zip(reranked, input_sets)
    )
    assert all(len(row["pre_reranked"]) == RERANK_DEPTH for row in reranked)
    output_path = OUTPUT_DIR / f"LitSearch.title_abstract.{retriever}.modernbert.reranked.jsonl"
    write_jsonl(output_path, reranked)
    run_records[retriever] = {
        "input": str(input_path.resolve()),
        "input_sha256": sha256_file(input_path),
        "output": str(output_path.resolve()),
        "output_sha256": sha256_file(output_path),
        "queries": len(rows),
        "documents": len(rows) * RERANK_DEPTH,
        "elapsed_seconds": elapsed,
        "queries_per_second": len(rows) / elapsed,
        "documents_per_second": len(rows) * RERANK_DEPTH / elapsed,
        "peak_vram": (
            torch.cuda.max_memory_reserved() / (1024 ** 3)
            if torch.cuda.is_available() else 0.0
        ),
    }
    print(retriever, json.dumps(run_records[retriever], indent=2))

In [ ]:
# Use the same metric implementation as every existing LitSearch run.
for record in run_records.values():
    command = [
        sys.executable,
        str(REPO_ROOT / "LitSearch" / "evaluate_results.py"),
        "--input",
        record["output"],
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

manifest = {
    "experiment": "modernbert_litsearch_frozen_top100",
    "model_directory": str(MODEL_DIR.resolve()),
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "rerank_depth": RERANK_DEPTH,
    "max_length": MAX_LENGTH,
    "predict_batch_size": PREDICT_BATCH_SIZE,
    "runs": run_records,
    "environment": reproducibility_manifest(),
}
write_json(OUTPUT_DIR / "litsearch_evaluation_manifest.json", manifest)

The `.metrics.json` files created beside each output contain the
table-ready overall R@20, R@5, and nDCG@20 values. Add these output
JSONL files to the paired-statistical-uncertainty configuration; do
not compute uncertainty from the three aggregate numbers alone.